# Busca de redução de parâmetros — RadioML 2018.01A

Treina as **8 candidatas** de redução com a mesma estratificação conjunta
(classe, SNR), o mesmo k-fold e o mesmo critério de parada da campanha local.

## O que a busca compara

A arquitetura convolucional fica **fixa** (`2L_32-64`); o que varia é o que
chega na camada densa — onde estão 99,9% dos parâmetros. Medido no ASK (3 classes):

| # | candidata | parâmetros | redução |
|---|---|---|---|
| 1 | `gap1` | 45.795 | 183× |
| 2 | `gap8_head128` | 77.027 | 109× |
| 3 | `gap4` | 144.099 | 58× |
| 4 | `gap8` | 275.171 | 30× |
| 5 | `6pools` | 595.427 | 14× |
| 6 | `head64` | 1.059.811 | 7,9× |
| 7 | `head128` | 2.108.643 | 4,0× |
| 8 | `baseline_2L_512` | 8.401.635 | 1× (controle) |

## Por que salvar no Drive

O Colab encerra a sessão num teto rígido de **~63 min** — medido em 77 sessões,
agrupadas em 62–64 min, independente de GPU, região ou crédito pago. Não há como
evitar.

Com os artefatos no Drive, isso deixa de importar: ao reabrir este notebook, o
treino **retoma do último checkpoint**, na época exata, com o estado dos
geradores aleatórios restaurado. Uma queda custa uma época.

## Fonte única

Este notebook **não** carrega uma cópia do motor de treino. Ele baixa
`busca_hp.py` e `resnet.py` do repositório, fixados no commit `f6f20e6`, e confere
antes de rodar que as condições de invariância estão todas presentes.

A versão anterior embutia as duas (1.788 linhas coladas). Cópia embutida
diverge: em 28/08/2026 ela estava sem a correção que restaura o RNG da GPU na
retomada — um `except: pass` engolia o erro, e o fold continuava com o gerador
não restaurado, em silêncio. A promessa de "retoma na época exata, com o estado
dos geradores restaurado" era falsa e nada acusava.


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CONFIGURAÇÃO — altere só este bloco
# ══════════════════════════════════════════════════════════════════════

REPO   = "Joammp/ML_Radio_Signal"
COMMIT = "f6f20e65aca3c606f571a5de2d76bef055c6334c"
                      # Commit FIXO, nao um branch: o codigo nao pode mudar
                      # debaixo de uma campanha em andamento. Para atualizar,
                      # troque pelo SHA novo -- a checagem de invariantes
                      # abaixo recusa um commit que tenha regredido.

ALVO = "ASK"          # "GROUP" | "ASK" | "PSK" | "APSK" | "QAM"
                      # GROUP = 19 digitais rotuladas pelo grupo (4 classes)

LR = 4.5e-4           # LR inicial. 4.5e-5 truncava o treino no teto de épocas;
                      # 1e-3 colapsa o QAM em 15/15 folds medidos.

USAR_DRIVE = True     # False = tudo em /content e some com a sessão

PASTA_DRIVE = "/content/drive/MyDrive/radioml_sessions"

EPOCAS_TETO = 1000    # alto de propósito: quem encerra é a parada por
                      # estagnação (25 épocas sem avançar 0,25 p.p.), não um
                      # número fixo que não sabe nada do treino.
CKPT_CADA = 1         # épocas entre checkpoints (0,6 MB cada nestes modelos)


In [ ]:
import os, sys

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE = PASTA_DRIVE
    os.makedirs(BASE, exist_ok=True)
    print("artefatos em:", BASE, "-- sobrevivem ao fim da sessao")
else:
    BASE = "/content/drive_cache/radioml_sessions"
    print("artefatos em:", BASE, "-- PERDIDOS quando a sessao cair")

try:
    import kagglehub
except ImportError:
    !pip install -q kagglehub
    import kagglehub

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  MOTOR DE TREINO — baixado do repositório, fixado no commit
# ══════════════════════════════════════════════════════════════════════
import hashlib, pathlib, urllib.request

RAW = "https://raw.githubusercontent.com/%s/%s/campanha/%s"

# As condições que precisam valer para os resultados serem comparáveis entre
# si. Se um commit futuro regredir qualquer uma, o notebook para AQUI, em vez
# de produzir números que parecem bons e não são comparáveis com os anteriores.
INVARIANTES = {
    "fold_seed = SEED + fold_i":
        "toda candidata parte da mesma inicialização em cada fold",
    "torch.backends.cudnn.deterministic = True":
        "kernels determinísticos",
    "torch.backends.cudnn.benchmark     = False":
        "sem autotune dependente de temporização",
    "torch.backends.cudnn.allow_tf32       = False":
        "convoluções em fp32 -- a L4 cairia para 10 bits de mantissa",
    "_byte_cpu":
        "a retomada restaura o RNG da GPU (senão o except engole o erro)",
    '"gpu"         : _GPU_NOME':
        "GPU e versões gravadas em cada fold, para auditar depois",
    "STRAT_BY_SNR":
        "estratificação conjunta (classe, SNR)",
}

for nome in ("resnet.py", "busca_hp.py"):
    dados = urllib.request.urlopen(RAW % (REPO, COMMIT, nome), timeout=120).read()
    pathlib.Path("/content", nome).write_bytes(dados)
    print("%-12s %6d bytes   sha256 %s"
          % (nome, len(dados), hashlib.sha256(dados).hexdigest()[:16]))

fonte = pathlib.Path("/content/busca_hp.py").read_text(encoding="utf-8")
faltando = [d for m, d in INVARIANTES.items() if m not in fonte]
if faltando:
    raise SystemExit("INVARIANTES AUSENTES no commit %s:\n  - %s"
                     % (COMMIT[:7], "\n  - ".join(faltando)))

print("\n%d invariantes conferidas   |   commit %s"
      % (len(INVARIANTES), COMMIT[:7]))

In [ ]:
# ====================================================================
#  EXECUCAO -- pode reexecutar quantas vezes quiser; retoma de onde parou
# ====================================================================
import os, runpy

os.environ["BUSCA_HP_ARGS"] = (
    "--grupos %s --lr %g --modelo reducao"
    " --epochs %d --ckpt-every %d --drive-base %s"
    % (ALVO, LR, EPOCAS_TETO, CKPT_CADA, BASE)
)
print("args:", os.environ["BUSCA_HP_ARGS"])
print()

runpy.run_path("/content/busca_hp.py", run_name="__main__")


## Ao reabrir depois de uma queda

Execute as células na ordem. A terceira e a quarta apenas regravam os scripts;
a última **retoma automaticamente**:

```
✅ N resultados anteriores  (N combinações concluídas)
Folds restantes: [...]
↺ Retomando da epoca N (best ate aqui X%, lr=...)
```

A primeira linha pula folds já concluídos; a terceira retoma um fold
interrompido no meio.

## Onde ficam os resultados

`<PASTA_DRIVE>/<ALVO>_reducao/`

| arquivo | conteúdo |
|---|---|
| `kfold_fold_results.json` | um registro por fold, com `history` época a época |
| `kfold_summary.json` | resumo por candidata |
| `ckpt_atual.pt` | checkpoint do fold em andamento (apagado ao concluir) |

Cada registro carrega os parâmetros sob os quais foi produzido (`lr`,
`stagnation_patience`, `min_delta`), então resultados de regimes diferentes
nunca se confundem.
